In [2]:
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, KFold

## Оптмизиаця признакового пространства

In [3]:
data = pd.read_csv("../data/processed_smoke_detector.csv")
X = data.drop(labels=["Fire Alarm"], axis=1)
y = data["Fire Alarm"]

In [4]:
from sklearn.feature_selection import SelectKBest

In [5]:
skb = SelectKBest(k=2)
X_skb = skb.fit_transform(X, y)
X_skb = pd.DataFrame(X_skb, columns=skb.get_feature_names_out())
X_skb.head()

,TVOC[ppb],Raw Ethanol
0,19.0,19951.0
1,1.0,19975.0
2,10.0,19955.0
3,10.0,19963.0
4,13.0,19958.0


In [6]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [7]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_skb, y, test_size=0.2, random_state=42)

## Sklearn MLP

In [7]:
import numpy as np
import optuna
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from scipy.stats import loguniform
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

### RandomSearch

In [8]:
param_dist = {
    'hidden_layer_sizes': [(32,), (64,), (32,32), (64,32), (64,64)],
    'activation': ['relu', 'tanh', 'logistic'],
    'solver': ['adam', 'sgd', 'lbfgs'],
    'alpha': loguniform(1e-3, 1e-1),
    'learning_rate_init': loguniform(1e-3, 0.1),
    'batch_size': [16, 32, 64, 128]
}

mlp = MLPClassifier(max_iter=100, random_state=42)

random_search = RandomizedSearchCV(mlp, param_dist, n_iter=10, cv=5, scoring='f1', n_jobs=-1)
random_search.fit(X_train_clf, y_train_clf)

print("RandomizedSearchCV results:")
print(f"Best params: {random_search.best_params_}")
print(f"Best CV accuracy: {random_search.best_score_:.4f}")
print(f"Test accuracy: {random_search.score(X_test_clf, y_test_clf):.4f}")


RandomizedSearchCV results:
Best params: {'activation': 'tanh', 'alpha': 0.010871651672879528, 'batch_size': 64, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.0015913652792979266, 'solver': 'lbfgs'}
Best CV accuracy: 0.9302
Test accuracy: 0.9231


## Optuna

In [ ]:
def objective(trial):
    n_layers = trial.suggest_int('n_layers', 1, 3)
    hidden_layer_sizes = []
    for i in range(n_layers):
        neurons = trial.suggest_int(f'neurons_layer{i}', 32, 256, step=32)
        hidden_layer_sizes.append(neurons)
    
    params = {
        'hidden_layer_sizes': tuple(hidden_layer_sizes),
        'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
        'solver': trial.suggest_categorical('solver', ['adam', 'sgd', 'lbfgs']),
        'alpha': trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 0.1, log=True),
        'batch_size': trial.suggest_int('batch_size', 16, 128, step=16),
        'max_iter': trial.suggest_int('max_iter', 200, 1000, step=100),
        'early_stopping': trial.suggest_categorical('early_stopping', [True, False]),
        'random_state': 42
    }
    
    model = MLPClassifier(**params)
    scores = cross_val_score(model, X_train_clf, y_train_clf, 
                           cv=5, scoring='accuracy', n_jobs=-1)
    return np.mean(scores)

study = optuna.create_study(direction='maximize',
                           sampler=optuna.samplers.TPESampler(),
                           pruner=optuna.pruners.HyperbandPruner())
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best trial:")
trial = study.best_trial
print(f"  Value (Accuracy): {trial.value:.4f}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

[I 2025-06-06 23:57:08,921] A new study created in memory with name: no-name-8e4e6ada-8cc1-4758-8a18-56519e8bf55d


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-06-06 23:57:10,115] Trial 0 finished with value: 0.7819498730294306 and parameters: {'n_layers': 1, 'neurons_layer0': 32, 'activation': 'relu', 'solver': 'lbfgs', 'alpha': 0.0907844730856697, 'learning_rate_init': 0.004465882251650786, 'batch_size': 96, 'max_iter': 900, 'early_stopping': True}. Best is trial 0 with value: 0.7819498730294306.
[I 2025-06-06 23:58:36,791] Trial 1 finished with value: 0.7819498730294306 and parameters: {'n_layers': 3, 'neurons_layer0': 256, 'neurons_layer1': 160, 'neurons_layer2': 32, 'activation': 'tanh', 'solver': 'sgd', 'alpha': 0.005889468298642918, 'learning_rate_init': 0.07015347343964871, 'batch_size': 64, 'max_iter': 600, 'early_stopping': False}. Best is trial 0 with value: 0.7819498730294306.
[I 2025-06-06 23:58:50,228] Trial 2 finished with value: 0.7819498730294306 and parameters: {'n_layers': 2, 'neurons_layer0': 160, 'neurons_layer1': 224, 'activation': 'tanh', 'solver': 'adam', 'alpha': 0.0013906916764791159, 'learning_rate_init': 0.

Trial 4 finished with value: 0.8668080103964328(f1) and parameters: 
-'n_layers': 3, 

-'neurons_layer0': 96, 

-'neurons_layer1': 224, 

-'neurons_layer2': 128, 

-'activation': 'relu', 

-'solver': 'adam', 

-'alpha': 0.00014161784924840714, 

-'learning_rate_init': 0.0007433050190842888, 

-'batch_size': 16, 

-'max_iter': 200, 

-'early_stopping': False

In [12]:
space = {
    'hidden_layer_sizes': hp.choice('hidden_layer_sizes', 
                    [(50,), (100,), (50,50), (100,50), (100,100)]),
    'activation': hp.choice('activation', ['relu', 'tanh', 'logistic']),
    'solver': hp.choice('solver', ['adam', 'sgd', 'lbfgs']),
    'alpha': hp.loguniform('alpha', np.log(1e-5), np.log(1e-1)),
    'learning_rate_init': hp.loguniform('learning_rate_init', np.log(1e-4), np.log(0.1)),
    'batch_size': hp.choice('batch_size', [16, 32, 64, 128]),
}

def objective_hyperopt(params):
    model_params = params.copy()
    
    model = MLPClassifier(
        **model_params,
        max_iter=100,
        random_state=42
    )
    
    score = cross_val_score(
        model, 
        X_train_clf, 
        y_train_clf, 
        cv=5, 
        scoring='f1',
        n_jobs=-1
    ).mean()
    
    return {'loss': -score, 'status': STATUS_OK, 'params': params}

trials = Trials()
best = fmin(
    fn=objective_hyperopt,
    space=space,
    algo=tpe.suggest,
    max_evals=20,
    trials=trials,
    rstate=np.random.default_rng(42)
)

best_trial = trials.best_trial
best_params = best_trial['result']['params']

print("\nHyperopt results:")
print(f"Best params: {best_params}")
print(f"Best accuracy: {-best_trial['result']['loss']:.4f}")

100%|██████████| 20/20 [04:16<00:00, 12.84s/trial, best loss: -0.9310719969681488]

Hyperopt results:
Best params: {'activation': 'logistic', 'alpha': 0.0006404758863588372, 'batch_size': 32, 'hidden_layer_sizes': (100, 100), 'learning_rate_init': 0.027685374484311354, 'solver': 'lbfgs'}
Best accuracy: 0.9311


## Keras + Tensor Flow FCNN

In [16]:
import numpy as np
import optuna
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers
from sklearn.model_selection import train_test_split, KFold

In [17]:
input_shape = X_train_clf.shape[1:]
num_classes = len(np.unique(y_train_clf))

### Optuna

In [24]:
def create_model_optuna(trial):
    n_layers = trial.suggest_int('n_layers', 1, 3)
    units = trial.suggest_int('units', 32, 256, step=32)
    dropout_rate = trial.suggest_float('dropout_rate', 0.0, 0.5, step=0.1)
    lr = trial.suggest_loguniform('lr', 1e-4, 1e-2)
    opt_name = trial.suggest_categorical('optimizer', ['adam', 'sgd', 'rmsprop'])

    model = keras.Sequential()
    model.add(layers.Input(shape=input_shape))
    for _ in range(n_layers):
        model.add(layers.Dense(units, activation='relu'))
        model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(num_classes, activation='softmax'))

    if opt_name == 'adam':
        optimizer = optimizers.Adam(learning_rate=lr)
    elif opt_name == 'sgd':
        optimizer = optimizers.SGD(learning_rate=lr)
    else:
        optimizer = optimizers.RMSprop(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def objective(trial):
    model = create_model_optuna(trial)
    history = model.fit(
        X_train_clf, y_train_clf,
        validation_split=0.2,
        epochs=20,
        batch_size=trial.suggest_int('batch_size', 16, 128, step=16),
        verbose=0
    )
    val_acc = history.history['val_accuracy'][-1]
    return val_acc

In [27]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5)
print("Best Optuna hyperparameters:", study.best_params)

[I 2025-06-07 11:05:25,918] A new study created in memory with name: no-name-42d9fd4d-187a-4e70-aa42-c8f3a4f6cf11
/var/folders/s6/j_4p__nd5kg5dvjxbj6y3ywh0000gn/T/ipykernel_2869/3301201888.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform('lr', 1e-4, 1e-2)
[I 2025-06-07 11:05:31,641] Trial 0 finished with value: 0.8910605907440186 and parameters: {'n_layers': 1, 'units': 192, 'dropout_rate': 0.5, 'lr': 0.00013392430763765447, 'optimizer': 'adam', 'batch_size': 96}. Best is trial 0 with value: 0.8910605907440186.
/var/folders/s6/j_4p__nd5kg5dvjxbj6y3ywh0000gn/T/ipykernel_2869/3301201888.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  l

Best Optuna hyperparameters: {'n_layers': 2, 'units': 96, 'dropout_rate': 0.0, 'lr': 0.0023238228301707764, 'optimizer': 'adam', 'batch_size': 96}


### Keras tuner

In [28]:
import keras_tuner as kt

In [29]:
def build_model_kt(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=input_shape))
    for i in range(hp.Int('n_layers', 1, 3)):
        model.add(layers.Dense(
            units=hp.Int(f'units_{i}', 32, 256, step=32),
            activation='relu'
        ))
        model.add(layers.Dropout(hp.Float(f'dropout_{i}', 0.0, 0.5, step=0.1)))
    model.add(layers.Dense(num_classes, activation='softmax'))

    optimizer_choices = hp.Choice('optimizer', ['adam', 'sgd', 'rmsprop'])
    lr = hp.Float('learning_rate', 1e-4, 1e-2, sampling='log')

    if optimizer_choices == 'adam':
        optimizer = optimizers.Adam(learning_rate=lr)
    elif optimizer_choices == 'sgd':
        optimizer = optimizers.SGD(learning_rate=lr)
    else:
        optimizer = optimizers.RMSprop(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [30]:
tuner = kt.RandomSearch(
    build_model_kt,
    objective='val_accuracy',
    max_trials=5, # 20
    executions_per_trial=1,
    directory='kt_dir',
    project_name='fcnn_clf'
)

tuner.search(
    X_train_clf, y_train_clf,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    verbose=1
)

best_hps_kt = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best KerasTuner hyperparameters:", best_hps_kt.values)



Trial 5 Complete [00h 00m 13s]
val_accuracy: 0.8978787660598755

Best val_accuracy So Far: 0.8978787660598755
Total elapsed time: 00h 01m 04s
Best KerasTuner hyperparameters: {'n_layers': 1, 'units_0': 128, 'dropout_0': 0.1, 'optimizer': 'adam', 'learning_rate': 0.0009005932092457443, 'units_1': 64, 'dropout_1': 0.2, 'units_2': 160, 'dropout_2': 0.4}


In [39]:
import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler
# from ray.tune.integration.keras import TuneReportCallback
from ray.train.tensorflow.keras import ReportCheckpointCallback

In [40]:
def ray_trainable(config):
    model = keras.Sequential()
    model.add(layers.Input(shape=input_shape))
    for _ in range(config['n_layers']):
        model.add(layers.Dense(config['units'], activation='relu'))
        model.add(layers.Dropout(config['dropout_rate']))
    model.add(layers.Dense(num_classes, activation='softmax'))

    opt_name = config['optimizer']
    lr = config['learning_rate']
    if opt_name == 'adam':
        optimizer = optimizers.Adam(learning_rate=lr)
    elif opt_name == 'sgd':
        optimizer = optimizers.SGD(learning_rate=lr)
    else:
        optimizer = optimizers.RMSprop(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        X_train_clf, y_train_clf,
        validation_split=0.2,
        epochs=20,
        batch_size=int(config['batch_size']),
        verbose=0,
        callbacks=[ReportCheckpointCallback()]
    )

search_space = {
    'n_layers': tune.randint(1, 4),
    'units': tune.choice([32, 64, 128, 256]),
    'dropout_rate': tune.uniform(0.0, 0.5),
    'optimizer': tune.choice(['adam', 'sgd', 'rmsprop']),
    'learning_rate': tune.loguniform(1e-4, 1e-2),
    'batch_size': tune.choice([16, 32, 64, 128])
}


In [48]:
scheduler = ASHAScheduler(
    metric='val_accuracy',
    mode='max',
    max_t=20,
    grace_period=5,
    reduction_factor=2
)

analysis = tune.run(
    ray_trainable,
    resources_per_trial={'cpu': 1, 'gpu': 0},
    config=search_space,
    num_samples=20,
    scheduler=scheduler,
    verbose=1
)

2025-06-07 12:17:21,391	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/Users/nikolajsemikin/ray_results/ray_trainable_2025-06-07_12-15-32' in 0.0158s.
2025-06-07 12:17:21,397	INFO tune.py:1041 -- Total run time: 108.83 seconds (108.80 seconds for the tuning loop).
